# Analisis Karakteristik Dataset - Adult Census Income

Notebook ini berisi eksplorasi data (EDA) untuk dataset **Adult Census Income**
(sumber: Kaggle / UCI Machine Learning Repository), digunakan untuk mengisi
https://www.kaggle.com/datasets/uciml/adult-census-income
Laporan Karakteristik Dataset.

**Isi notebook:**
1. Informasi umum dataset
2. Deskripsi fitur (data dictionary)
3. Analisis kualitas data (missing values, duplikat, outlier)
4. Statistik deskriptif
5. Distribusi target & korelasi antar fitur


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = pd.read_csv('adult.csv')
df.head()


## 1. Informasi Umum Dataset

In [ ]:
print("Jumlah baris :", df.shape[0])
print("Jumlah kolom :", df.shape[1])
print()
df.info()


In [ ]:
# Tabel ringkasan tipe data per kolom
tipe_df = pd.DataFrame({
    'Kolom': df.columns,
    'Tipe Data': df.dtypes.astype(str).values,
    'Contoh Nilai': [df[c].dropna().iloc[0] for c in df.columns]
})
tipe_df


## 2. Deskripsi Fitur (Data Dictionary)

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = ['age', 'fnlwgt', 'education.num',
            'capital.gain', 'capital.loss', 'hours.per.week']

print("Fitur numerik  :", num_cols)
print("Fitur kategorikal:", cat_cols)

In [ ]:
# Tabel jumlah kategori unik untuk tiap fitur kategorikal
unik_df = pd.DataFrame({
    'Kolom': cat_cols,
    'Jumlah Kategori Unik': [df[c].nunique() for c in cat_cols],
    'Contoh Kategori': [list(df[c].unique()[:5]) for c in cat_cols]
})
unik_df


## 3. Analisis Kualitas Data

### 3.1 Missing Values

In [ ]:
# Cek NaN literal
print("Missing values (NaN literal):")
print(df.isna().sum()[df.isna().sum() > 0])


In [ ]:
# Dataset ini menandai data hilang dengan simbol '?', bukan NaN
missing_tanda = {}
for col in cat_cols:
    cnt = (df[col] == '?').sum()
    if cnt > 0:
        missing_tanda[col] = cnt

missing_df = pd.DataFrame({
    'Kolom': missing_tanda.keys(),
    'Jumlah Missing (\'?\')': missing_tanda.values(),
    'Persentase (%)': [round(v/len(df)*100, 2) for v in missing_tanda.values()]
})
missing_df


In [ ]:
# Visualisasi persentase missing value per kolom
plt.figure(figsize=(7,4))
sns.barplot(data=missing_df, x='Kolom', y='Persentase (%)', hue='Kolom', palette='Reds_r', legend=False)
plt.title('Persentase Missing Value per Kolom')
plt.ylabel('Persentase (%)')
plt.tight_layout()
plt.show()


### 3.2 Data Duplikat

In [ ]:
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat penuh: {jumlah_duplikat} ({jumlah_duplikat/len(df)*100:.2f}%)")


### 3.3 Outlier (metode IQR)

In [ ]:
outlier_summary = []
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outlier = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    outlier_summary.append({
        'Fitur': col,
        'Jumlah Outlier': n_outlier,
        'Persentase (%)': round(n_outlier/len(df)*100, 2),
        'Batas Bawah': round(lower, 2),
        'Batas Atas': round(upper, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Persentase (%)', ascending=False)
outlier_df


In [ ]:
# Visualisasi outlier dengan boxplot untuk tiap fitur numerik
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'Boxplot: {col}')

plt.tight_layout()
plt.show()


## 4. Statistik Deskriptif (Fitur Numerik)

In [ ]:
desc = df[num_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
desc.columns = ['Mean', 'Std Dev', 'Min', '25% (Q1)', '50% (Q2)', '75% (Q3)', 'Max']
desc.style.background_gradient(cmap='Blues', axis=0).format('{:,.2f}')


In [ ]:
# Histogram distribusi tiap fitur numerik
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Distribusi: {col}')

plt.tight_layout()
plt.show()


## 5. Analisis Distribusi Target & Korelasi

### 5.1 Distribusi Variabel Target (income)

In [ ]:
target_counts = df['income'].value_counts()
target_pct = df['income'].value_counts(normalize=True) * 100

target_df = pd.DataFrame({
    'Jumlah': target_counts,
    'Persentase (%)': target_pct.round(2)
})
target_df


In [ ]:
# Bar chart distribusi target
plt.figure(figsize=(6,4))
colors = ['#4C72B0', '#DD8452']
ax = sns.countplot(data=df, x='income', hue='income', palette=colors, legend=False)
plt.title('Distribusi Variabel Target (income)')
plt.xlabel('Kategori Income')
plt.ylabel('Jumlah')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom')

plt.tight_layout()
plt.show()


### 5.2 Korelasi Antar Fitur Numerik & Target

In [ ]:
# ubah target jadi biner (1 = '>50K', 0 = '<=50K') agar bisa dihitung korelasinya
df['income_bin'] = (df['income'] == '>50K').astype(int)

corr = df[num_cols + ['income_bin']].corr()

print("Korelasi terhadap target (diurutkan):")
corr['income_bin'].sort_values(ascending=False).drop('income_bin')


In [ ]:
# Heatmap matriks korelasi
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Heatmap Korelasi Antar Fitur Numerik & Target')
plt.tight_layout()
plt.show()


## 6. Ringkasan Temuan

- Dataset berjumlah **32.561 baris x 15 kolom**, terdiri dari 14 fitur dan 1 target (`income`).
- Missing value disimbolkan dengan `'?'` pada kolom `workclass`, `occupation`, dan `native.country`.
- Terdapat **24 baris duplikat** yang perlu dihapus.
- Fitur `hours.per.week`, `capital.gain`, dan `capital.loss` memiliki proporsi outlier cukup tinggi (secara alami, bukan kesalahan input).
- Target **tidak seimbang**: 75,92% `<=50K` vs 24,08% `>50K`.
- `education.num` memiliki korelasi tertinggi terhadap target (r ≈ 0,34), sedangkan `fnlwgt` praktis tidak berkorelasi (r ≈ -0,01).
